# Controller Interface and Registry

**Sistemas de Decision y Control Inteligente (SDCI)**  
**Universidad de los Andes, 2026**

Copyright (c) 2026 Leffer Trochez

---

All controller-related configuration and code is isolated in `config/controller/`.

The fixed runtime interface is:

- Inputs: `T_ref`, `T_error`, `T_meas`
- Output: `duty_cycle`

Each controller must implement `reset()` and `step(T_ref, T_error, T_meas)`.


In [ ]:
import numpy as np


class ControllerInterfaceError(RuntimeError):
    """Raised when a controller violates the required project interface."""


def create_controller(controller_id, CTRL):
    """
    Create the selected controller.

    Every controller must implement:
        reset()
        step(T_ref, T_error, T_meas) -> duty_cycle
    """

    controller_id = int(controller_id)

    factories = {
        1: ("ON_OFF", create_on_off_controller),
        2: ("FUZZY", create_fuzzy_controller),
        3: ("MPC", create_mpc_controller),
        4: ("ESC", create_esc_controller),
        5: ("REPLICATOR", create_replicator_controller),
    }

    if controller_id not in factories:
        raise ControllerInterfaceError(
            "Controller ID must be an integer from 1 to 5."
        )

    parameter_key, factory = factories[controller_id]

    try:
        controller = factory(CTRL[parameter_key])
    except NotImplementedError as exc:
        raise ControllerInterfaceError(
            f"{parameter_key} controller is not implemented."
        ) from exc
    except Exception as exc:
        raise ControllerInterfaceError(
            f"Could not initialize {parameter_key} controller: {exc}"
        ) from exc

    if not hasattr(controller, "reset") or not callable(controller.reset):
        raise ControllerInterfaceError(
            f"{parameter_key} must implement reset()."
        )

    if not hasattr(controller, "step") or not callable(controller.step):
        raise ControllerInterfaceError(
            f"{parameter_key} must implement step(T_ref, T_error, T_meas)."
        )

    return controller


def controller_parameters(MODEL):
    """Load all controller parameters and register the common interface."""

    CTRL = {}

    # SHARED PARAMETERS
    CTRL["shared"] = {
        "numberOfRooms": 4,
        "dMin": 0.0,
        "dMax": 1.0,
        "sampleTime": 1.0,
    }

    # ACTUATOR POWER AVAILABLE TO CONTROLLER CONFIGURATION
    rated_power = float(MODEL["ACTUATOR"]["P_rated"])
    if not np.isfinite(rated_power) or rated_power <= 0:
        raise ValueError(
            "Actuator rated power must be a positive finite scalar."
        )
    CTRL["shared"]["maxRoomPower"] = rated_power

    # CONTROLLER PARAMETERS + IMPLEMENTATIONS
    CTRL["ON_OFF"] = on_off_parameters(CTRL["shared"])
    CTRL["FUZZY"] = fuzzy_parameters(CTRL["shared"])
    CTRL["MPC"], MPC_PLANT, MPC_CONTROLLER = mpc_parameters(
        CTRL["shared"], MODEL
    )
    CTRL["ESC"] = esc_parameters(CTRL["shared"])
    CTRL["REPLICATOR"] = replicator_parameters(CTRL["shared"])

    # METADATA
    CTRL["names"] = [
        "ON_OFF",
        "FUZZY",
        "MPC",
        "ESC",
        "REPLICATOR",
    ]
    CTRL["displayNames"] = [
        "ON/OFF",
        "Fuzzy Logic",
        "Model Predictive Control",
        "Extremum Seeking Control",
        "Replicator Dynamics",
    ]

    # FIXED CONTROLLER INTERFACE
    CTRL["interface"] = {
        "inputs": ["T_ref", "T_error", "T_meas"],
        "output": "duty_cycle",
        "channels": 4,
    }
    CTRL["createController"] = create_controller

    # Compatibility aliases used by the project context.
    CTRL_REPLICATOR = CTRL["REPLICATOR"]

    print("Controller configuration loaded.")
    print("  Student workspace: config/controller/")
    print("  Interface: T_ref, T_error, T_meas -> duty_cycle")

    return CTRL, CTRL_REPLICATOR, MPC_PLANT, MPC_CONTROLLER
